In [ ]:
!pip install -q transformers datasets torch scikit-learn


In [ ]:
from datasets import load_dataset

dataset = load_dataset("sms_spam", split="train")
print(dataset[0])


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/359k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5574 [00:00<?, ? examples/s]

{'sms': 'Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...\n', 'label': 0}


In [ ]:
dataset = dataset.train_test_split(test_size=0.2, seed=42)
train_ds = dataset["train"]
test_ds = dataset["test"]


In [ ]:
from transformers import AutoTokenizer

model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(batch["sms"], truncation=True, padding="max_length", max_length=128)

train_ds = train_ds.map(tokenize, batched=True)
test_ds = test_ds.map(tokenize, batched=True)

train_ds = train_ds.remove_columns(["sms"])
test_ds = test_ds.remove_columns(["sms"])
train_ds.set_format("torch")
test_ds.set_format("torch")


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/4459 [00:00<?, ? examples/s]

Map:   0%|          | 0/1115 [00:00<?, ? examples/s]

In [ ]:
from transformers import AutoModelForSequenceClassification

id2label = {0: "HAM", 1: "SPAM"}
label2id = {"HAM": 0, "SPAM": 1}

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
from transformers import TrainingArguments, IntervalStrategy

training_args = TrainingArguments(
    output_dir="./spam_model",
    # evaluation_strategy=IntervalStrategy.EPOCH, # This argument is causing a TypeError in the current transformers version
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=4,
    weight_decay=0.01,
    logging_steps=50,
    report_to="none"
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds
)

trainer.train()


Step,Training Loss
50,0.271200
100,0.040500
150,0.038100
200,0.043100
250,0.034100
300,0.045500
350,0.018600
400,0.026500
450,0.017500
500,0.002900


TrainOutput(global_step=1116, training_loss=0.027321779074007145, metrics={'train_runtime': 406.6937, 'train_samples_per_second': 43.856, 'train_steps_per_second': 2.744, 'total_flos': 1173212195850240.0, 'train_loss': 0.027321779074007145, 'epoch': 4.0})

In [ ]:
from sklearn.metrics import classification_report

preds = trainer.predict(test_ds)
y_pred = preds.predictions.argmax(axis=1)
y_true = test_ds["label"]

print(classification_report(y_true, y_pred, target_names=["HAM", "SPAM"]))


              precision    recall  f1-score   support

         HAM       0.99      0.99      0.99       966
        SPAM       0.97      0.97      0.97       149

    accuracy                           0.99      1115
   macro avg       0.98      0.98      0.98      1115
weighted avg       0.99      0.99      0.99      1115



In [ ]:
import torch.nn.functional as F

def predict_spam_debug(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    outputs = model(**inputs)
    probs = F.softmax(outputs.logits, dim=1).detach().cpu().numpy()[0]

    return {
        "text": text,
        "HAM_prob": round(probs[0], 3),
        "SPAM_prob": round(probs[1], 3),
        "prediction": model.config.id2label[probs.argmax()]
    }


In [ ]:
print(predict_spam_debug("WIN CASH NOW!!! Click http://bit.ly"))
print(predict_spam_debug("Hey, are you coming to class today?"))
print(predict_spam_debug("Congratulations! You won a free iPhone"))
print(predict_spam_debug("Hi, are we meeting tomorrow?"))


{'text': 'WIN CASH NOW!!! Click http://bit.ly', 'HAM_prob': np.float32(0.001), 'SPAM_prob': np.float32(0.999), 'prediction': 'SPAM'}
{'text': 'Hey, are you coming to class today?', 'HAM_prob': np.float32(1.0), 'SPAM_prob': np.float32(0.0), 'prediction': 'HAM'}
{'text': 'Congratulations! You won a free iPhone', 'HAM_prob': np.float32(0.0), 'SPAM_prob': np.float32(1.0), 'prediction': 'SPAM'}
{'text': 'Hi, are we meeting tomorrow?', 'HAM_prob': np.float32(1.0), 'SPAM_prob': np.float32(0.0), 'prediction': 'HAM'}
